# Preprocessing — BC_A&A_with_ATD.csv

Simple, opinionated cleaning pipeline:

| Step | Action |
|------|---------|
| 1 | Load raw CSV, treating `\\N` (MySQL NULL) as `NaN` |
| 2 | Drop every row that contains **any** null / `\\N` value |
| 3 | Drop rows where **ATD > 180 min** (>3 h — data errors / extreme outliers) |
| 4 | Save to `data/processed/preprocessed.parquet` |

In [ ]:
import pandas as pd
from pathlib import Path

RAW_PATH     = Path('../data/raw/BC_A&A_with_ATD.csv')
RAW_OUT_PATH = Path('../data/processed/raw.parquet')
OUT_PATH     = Path('../data/processed/preprocessed.parquet')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

ATD_MAX_MIN = 180  # drop deliveries longer than 3 hours

## 1 · Load raw data

`\\N` is MySQL's NULL export representation — pass it as `na_values` so pandas treats it as `NaN` from the start.

In [ ]:
df = pd.read_csv(
    RAW_PATH,
    na_values=['\\N', 'NULL', ''],
    parse_dates=[
        'restaurant_offered_timestamp_utc',
        'order_final_state_timestamp_local',
        'eater_request_timestamp_local',
    ],
    dtype={
        'region': 'category',
        'territory': 'category',
        'country_name': 'category',
        'courier_flow': 'category',
        'geo_archetype': 'category',
        'merchant_surface': 'category',
    },
)

print(f'Raw rows    : {len(df):,}')
print(f'Columns     : {df.shape[1]}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print()

null_counts = df.isnull().sum()
print('Null counts per column (raw):')
print(null_counts[null_counts > 0].to_string())

In [ ]:
# Save raw data as parquet (preserves types, much faster to reload)
df.to_parquet(RAW_OUT_PATH, index=False, engine='pyarrow')

raw_size_mb = RAW_OUT_PATH.stat().st_size / 1024**2
print(f'Saved raw parquet to : {RAW_OUT_PATH}')
print(f'Size                 : {raw_size_mb:.1f} MB')

## 2 · Drop rows with any null / `\\N` value

In [ ]:
n_before = len(df)
df = df.dropna()
n_dropped_null = n_before - len(df)

print(f'Rows dropped (any null) : {n_dropped_null:,}  ({n_dropped_null / n_before * 100:.2f}%)')
print(f'Rows remaining          : {len(df):,}')

## 3 · Drop rows where ATD > 180 min (>3 h)

In [ ]:
n_before = len(df)
df = df[df['ATD'] <= ATD_MAX_MIN]
n_dropped_atd = n_before - len(df)

print(f'Rows dropped (ATD > {ATD_MAX_MIN} min) : {n_dropped_atd:,}  ({n_dropped_atd / n_before * 100:.2f}%)')
print(f'Rows remaining                  : {len(df):,}')
print()
print('ATD summary after filtering:')
print(df['ATD'].describe(percentiles=[.25, .5, .75, .95, .99]).round(2).to_string())

## 4 · Save to parquet

In [ ]:
df.to_parquet(OUT_PATH, index=False, engine='pyarrow')

size_mb = OUT_PATH.stat().st_size / 1024**2
print(f'Saved to : {OUT_PATH}')
print(f'Shape    : {df.shape}')
print(f'Size     : {size_mb:.1f} MB')
print(f'Columns  : {list(df.columns)}')